# Automated Code Review Board (Group 14)
### Course: CSE476 - Agentic AI and Intelligent Automation
**Coordination Architecture:** Fan-Out / Fan-In with Reviewer Manager  
**Framework:** Microsoft Semantic Kernel  
**Model Lane:** Groq (`openai/gpt-oss-120b`)  
**Group-of-3 Add-on:** Fix-Suggestion Agent (Auto-generates patch for top defect)

---
## System Workflow Overview
1. **Diff Intake & Metrics (`DiffAgent`):** Parses git diff, computes churn, files touched, and risk footprint.
2. **Parallel Fan-out Analysis:**
   - **`StyleAgent`:** Verifies PEP 8 compliance, docstring presence, and McCabe cyclomatic complexity.
   - **`SecurityAgent`:** Scans for hardcoded credentials, API keys, SQL injections, and dangerous APIs (`eval`, `exec`).
   - **`TestCoverageAgent`:** Verifies test file presence and inspects test suites for assertions exercising changed logic.
3. **Fan-in Adjudication (`ReviewerManager`):** Deduplicates findings, prioritizes by severity (`CRITICAL` -> `INFO`), and renders `approve` or `request-changes`.
4. **Group-of-3 Fix Suggestion (`FixSuggestionAgent`):** Synthesizes a concrete unified git patch for the highest-severity defect.


In [1]:
import os
import asyncio
from dotenv import load_dotenv
from core.orchestrator import CodeReviewOrchestrator
from core.change_state import Severity
from tests.sample_diffs import CLEAN_DIFF, SECURITY_VULN_DIFF, COMPLEXITY_AND_MISSING_TESTS_DIFF

load_dotenv()
print("Environment initialized.")
print("Model lane:", os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"))
orchestrator = CodeReviewOrchestrator()


Environment initialized.
Model lane: openai/gpt-oss-120b


In [2]:
def display_review_trace(state):
    print("=" * 80)
    verdict_str = state.merge_verdict.upper() if state.merge_verdict else "PENDING"
    print(f"DIFF ID: {state.diff_id} | VERDICT: {verdict_str}")
    print("=" * 80)
    print(f"\nExecutive Summary:\n  {state.summary}")
    print(f"\nReviewer Manager Rationale:\n  {state.verdict_rationale}")
    
    print(f"\nTotal Priority Findings: {len(state.priority_findings)}")
    for idx, f in enumerate(state.priority_findings, 1):
        sev_color = {
            'CRITICAL': '[CRITICAL]',
            'HIGH': '[HIGH]',
            'MEDIUM': '[MEDIUM]',
            'LOW': '[LOW]',
            'INFO': '[INFO]'
        }.get(f.severity.value, f'[{f.severity.value}]')
        line_info = f":{f.line_number}" if f.line_number else ""
        print(f"  {idx}. {sev_color} {f.rule_id} ({f.agent}) -> {f.file_path}{line_info}")
        print(f"     Description: {f.description}")
        if f.suggestion:
            print(f"     Suggestion:  {f.suggestion}")
            
    if state.suggested_patch:
        print("\n" + "-" * 80)
        print("FIX-SUGGESTION AGENT: PROPOSED UNIFIED PATCH")
        print("-" * 80)
        print(state.suggested_patch)
        
    if state.warnings:
        print("\n" + "!" * 80)
        print("HANDLED FAILURES / WARNINGS:")
        for w in state.warnings:
            print(f"  * {w}")
    print("=" * 80 + "\n")


## Scenario 1: Clean, Well-Tested Code Change
A pull request introducing a statistical variance utility with full docstrings, PEP 8 styling, and an accompanying unit test file.
**Expected Verdict:** `approve` (Zero blocking issues).

In [3]:
# Execute Scenario 1
state_clean = await orchestrator.review_code_change(
    raw_diff=CLEAN_DIFF,
    diff_id="DEMO-SCENARIO-1-CLEAN"
)
display_review_trace(state_clean)


DIFF ID: DEMO-SCENARIO-1-CLEAN | VERDICT: APPROVE

Executive Summary:
  The change introduces a new **src/math_utils.py** module that implements `calculate_variance`, a function computing the sample variance of a list of floats and raising a `ValueError` when fewer than two values are provided. Corresponding unit tests are added in **tests/test_math_utils.py** to verify correct variance calculation for a typical dataset and proper error handling for insufficient input. No existing code is altered, and the modification carries a low risk footprint.

Reviewer Manager Rationale:
  The addition of **src/math_utils.py** introduces a well‑encapsulated, pure‑function implementation of sample variance with clear error handling, and the accompanying tests in **tests/test_math_utils.py** confirm both correct computation and proper exception raising, representing a low‑risk, self‑contained enhancement that does not affect existing functionality. Although the test functions lack docstrings (STY‑00

## Scenario 2: Critical Security Flaws & Leaked Credentials
A pull request committing an exposed AWS Secret Key and a SQL injection vulnerability via raw string formatting.
**Expected Verdict:** `request-changes`, CRITICAL severity findings, and an automated patch from the Fix-Suggestion Agent.

In [4]:
# Execute Scenario 2
state_sec = await orchestrator.review_code_change(
    raw_diff=SECURITY_VULN_DIFF,
    diff_id="DEMO-SCENARIO-2-SECURITY"
)
display_review_trace(state_sec)


DIFF ID: DEMO-SCENARIO-2-SECURITY | VERDICT: REQUEST-CHANGES

Executive Summary:
  The change adds a hard‑coded AWS secret access key constant and introduces a new helper `query_user_profile` that retrieves a user’s `id`, `username`, and `email` from the SQLite `users` table. The function builds its SELECT statement by directly interpolating the supplied `user_id` into an f‑string, then executes the raw query and returns the first row. This modification expands the service’s API surface to include profile look‑ups while embedding sensitive credentials in source code.

Reviewer Manager Rationale:
  The proposed changes introduce a hard‑coded AWS Access Key ID and Secret Access Key, directly violating secret‑management best practices, and the new `query_user_profile` function builds its SQL statement via an f‑string, creating a high‑risk SQL‑injection vector. Additionally, the modification adds public API functionality without accompanying unit tests, leaving the new behavior unverified.

## Scenario 3: Missing Tests, High Complexity & Honest Failure Handling
A pull request introducing high cyclomatic complexity, zero unit tests, and an intentionally unparseable syntax fragment.
**Expected Handling:** AST parser catches `SyntaxError` honestly and falls back gracefully without crashing; missing tests and complexity are flagged.
**Expected Verdict:** `request-changes`.

In [5]:
# Execute Scenario 3
state_failure = await orchestrator.review_code_change(
    raw_diff=COMPLEXITY_AND_MISSING_TESTS_DIFF,
    diff_id="DEMO-SCENARIO-3-FAILURE-HANDLING"
)
display_review_trace(state_failure)


DIFF ID: DEMO-SCENARIO-3-FAILURE-HANDLING | VERDICT: REQUEST-CHANGES

Executive Summary:
  A new module `services/order_router.py` introduces a `route_order` function that determines an order’s dispatch path through a series of nested conditionals based on order type, VIP status, inventory, destination country, coupon presence, and fraud score. The function returns specific routing strings (e.g., `priority_dispatch`, `fraud_review`, `standard_express`, etc.) and falls back to generic options for non‑express or unavailable stock cases. An intentionally malformed block is appended at the end to trigger a syntax error for testing purposes.

Reviewer Manager Rationale:
  The submitted `services/order_router.py` introduces substantial routing logic but lacks any accompanying unit tests, leaving the new functionality unverified and increasing the risk of regressions in critical order‑processing pathways. Moreover, the file contains a deliberate syntax error that prevents the module from bein

## Append-Only Audit Run Log Inspection
Every agent execution, tool call, and verdict is preserved in the persistent `logs/audit_run_log.jsonl` audit trail.

In [6]:
from core.run_log import audit_logger
entries = audit_logger.read_all_entries()
print(f"Total entries in audit log: {len(entries)}")
print("\nLast 5 audit log entries:")
for e in entries[-5:]:
    print(f"[{e['timestamp']}] {e['event_type']} - Agent: {e['agent']} (Diff: {e['diff_id']})")


Total entries in audit log: 180

Last 5 audit log entries:
[2026-11-15T16:51:50.008951+00:00] DECISION - Agent: ReviewerManager (Diff: DEMO-SCENARIO-3-FAILURE-HANDLING)
[2026-11-15T16:51:50.009734+00:00] TOOL_CALL - Agent: FixSuggestionAgent (Diff: DEMO-SCENARIO-3-FAILURE-HANDLING)
[2026-11-15T16:51:51.559698+00:00] TOOL_CALL - Agent: FixSuggestionAgent (Diff: DEMO-SCENARIO-3-FAILURE-HANDLING)
[2026-11-15T16:51:51.560459+00:00] AGENT_COMPLETE - Agent: FixSuggestionAgent (Diff: DEMO-SCENARIO-3-FAILURE-HANDLING)
[2026-11-15T16:51:51.560692+00:00] WORKFLOW_COMPLETE - Agent: Orchestrator (Diff: DEMO-SCENARIO-3-FAILURE-HANDLING)
